# Sizing an electrified industrial heat supply behind a flexible connection agreement

Thin driver notebook. **All logic lives in the `fcaheat` package** — edit the package, not this
notebook. This file is for exploring results and building figures interactively.

Research question: *given a flexible connection agreement with a specified restriction structure,
how must the electrified heat supply be sized?*

The connection is a time series, not a scalar: $p_g(t) \le P_\mathrm{limit}(t)$, built from the
contract parameters that § 17 Abs. 2b EnWG requires the agreement to state.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from fcaheat import (CFG, load_inputs, run_batch, add_relative_kpis, run_contract_space,
                     run_sensitivity_oat, run_mpc_all_sites, export_tables, export_metadata)
from fcaheat import figures as F

CFG.update(years=[2024], months=None, resolution="1h")   # 15min + [2023,2024,2025] for final runs
CFG["fcas"] = ["FCA_FIRM", "FCA_WINDOW", "FCA_DYNAMIC", "FCA_TDTR", "FCA_UPGRADE"]
CFG

## 1 · Data

The workbook is the single source of truth. Validation runs on load and fails loudly.

In [ ]:
INP = load_inputs()
INP.sites[["site_id", "site_name", "sector", "P_grid_exist_MW", "load_factor",
           "heat_to_power", "T_supply_C", "atypical_eligible"]].round(3)

### 1.1 · Check this before any production run

Plot load factor against heat-to-power ratio. If the five sites cluster, the multi-site framing
does not hold and the paper needs restructuring around one site with a parameter sweep.

In [ ]:
F.fig_demand_overview(INP)
F.fig_duration_curves(INP)

## 2 · Connection regimes

What each agreement actually permits, over one week.

In [ ]:
F.fig_limit_profile(INP, INP.sites['site_id'].iloc[0])
INP.fca[["fca_id", "type", "P_static_rel", "P_flex_rel", "restricted_hours",
         "max_curtail_h_per_a", "notice_h", "netzentgelt_discount"]]

## 3 · Sizing runs

One LP per (site, storage configuration, connection regime).

In [ ]:
KPI, TS = run_batch(INP, sites=CFG["sites"], tag="main")
KPIX = add_relative_kpis(KPI)
KPIX[["site", "scenario", "fca", "feasible", "unserved_share_pct", "Qhp_MW", "Peb_MW",
      "Etes_MWh", "Ebes_MWh", "peak_grid_MW", "restricted_share",
      "binding_restricted_share", "restriction_bite_share"]].round(3)

## 4 · Figures

`restricted_share` is what the operator reserves; `restriction_bite_share` is what actually
constrains the plant. The gap between them is the negotiating argument.

In [ ]:
F.fig_feasibility_matrix(KPIX)
F.fig_storage_vs_regime(KPIX)
F.fig_kpi_comparison(KPIX)
F.fig_cost_stack(KPIX)

In [ ]:
site0 = KPIX["site"].iloc[0]
F.fig_dispatch(TS, site0, "S4_TES_BES", "FCA_WINDOW")
F.fig_grid_duration(TS, site0, inp=INP)

## 5 · Contract space

Inverted question: the minimum uplift and the maximum restriction width a given design can live
with. This is what a plant takes into the negotiation.

In [ ]:
CS = run_contract_space(INP, sites=CFG["sites"], scenarios=("S4_TES_BES",))
CS

## 6 · Rolling-horizon operation — the value of notice

Same hardware, two contracts: restrictions visible only within the response time versus notified
a day ahead. The difference is attributable purely to contract design.

In [ ]:
MPC = run_mpc_all_sites(INP, KPIX, fca="FCA_DYNAMIC", notices=(0.25, 24.0), seeds=(0,))
F.fig_notice_value(MPC)
MPC[["site", "notice_h", "unserved_MWh", "foresight_gap_MWh", "violations"]].round(2)

## 7 · Sensitivity

In [ ]:
SENS = run_sensitivity_oat(INP, site=KPIX["site"].iloc[0], scenario="S4_TES_BES",
                           fca="FCA_WINDOW", resolution="1h")
F.fig_tornado(SENS, "Etes_MWh", "TES size [MWh]")

## 8 · Export

In [ ]:
T2, T3, T4 = export_tables(KPIX)
export_metadata(INP)
T4